# MacroMind — Three-Variant Ablation Evaluation

Compares V1 (Baseline), V2 (RAG), and V3 (RAG + Constraint-Aware Reranking) across 5 user profiles.
Runs fully offline — no LLM API key required.

In [1]:
import random
import sys
import os

# Ensure repo root is on the path
sys.path.insert(0, os.path.abspath('..'))

from sentence_transformers import SentenceTransformer
from src.config import EMBEDDING_MODEL
from src.retriever import get_or_create_collection, semantic_search, UserConstraints
from src.reranker import rerank, RankedResult
from src.data_pipeline import load_cache
from src.evaluator import evaluate_variant, build_results_dataframe

random.seed(42)

# Load shared resources
model = SentenceTransformer(EMBEDDING_MODEL)
_, collection = get_or_create_collection()
nutrition_cache = load_cache()

print(f"{collection.count()} recipes indexed | {len(nutrition_cache)} ingredients cached")

10064 recipes indexed | 30 ingredients cached


In [2]:
# ── Test Profiles ──────────────────────────────────────────────────────────────
# relevant_ids: curated recipe IDs manually annotated as good matches per profile.
# Annotated after inspecting semantic retrieval results against recipe tags.

test_profiles = [
    {
        "name": "Body Recomp",
        "constraints": UserConstraints(
            calories=2000, protein_g=150, carbs_g=200, fat_g=65, budget_usd=12
        ),
        # High-protein meal-prep recipes from curated set
        "relevant_ids": [
            "r001", "r003", "r006", "r008", "r010", "r014", "r019",
            "r031", "r033", "r036", "r040", "r041", "r045", "r049",
            "r052", "r056", "r061", "r065",
        ],
    },
    {
        "name": "Muscle Bulk",
        "constraints": UserConstraints(
            calories=2800, protein_g=180, carbs_g=320, fat_g=80, budget_usd=16
        ),
        # High-calorie, high-protein, or balanced recipes
        "relevant_ids": [
            "r006", "r010", "r011", "r019", "r020", "r031", "r033",
            "r036", "r041", "r045", "r047", "r049", "r056", "r058",
            "r060", "r061",
        ],
    },
    {
        "name": "Vegetarian Weight Loss",
        "constraints": UserConstraints(
            calories=1600, protein_g=80, carbs_g=180, fat_g=50, budget_usd=9,
            dietary_tags=["vegetarian"]
        ),
        # Vegetarian-tagged curated recipes (lower calorie preference)
        "relevant_ids": [
            "r004", "r007", "r012", "r013", "r017", "r021", "r022",
            "r023", "r024", "r025", "r026", "r028", "r029", "r030",
            "r034", "r037", "r038", "r042", "r043", "r044", "r046",
            "r048", "r051", "r052", "r053", "r055", "r057", "r059",
            "r060", "r062", "r063", "r064",
        ],
    },
    {
        "name": "Budget Meal Prep",
        "constraints": UserConstraints(
            calories=1900, protein_g=100, carbs_g=230, fat_g=60, budget_usd=7
        ),
        # Budget-tagged and affordable curated recipes
        "relevant_ids": [
            "r002", "r004", "r007", "r013", "r021", "r022", "r026",
            "r028", "r029", "r030", "r032", "r034", "r038", "r045",
            "r047", "r055", "r056", "r057", "r059", "r060", "r061",
            "r063",
        ],
    },
    {
        "name": "Keto / Low Carb",
        "constraints": UserConstraints(
            calories=1800, protein_g=130, carbs_g=30, fat_g=120, budget_usd=14,
            dietary_tags=["keto"]
        ),
        # Keto-tagged curated recipes (low-carb, high-fat)
        "relevant_ids": [
            "r015", "r016", "r017", "r018", "r039", "r050",
            "r054", "r065",
        ],
    },
]

print(f"Test profiles loaded: {len(test_profiles)}")

Test profiles loaded: 5


In [3]:
# ── Helper: wrap random sample as RankedResult list ────────────────────────────

def random_baseline(collection, n=5):
    """V1: sample n random recipes, no retrieval."""
    total = collection.count()
    offset = random.randint(0, max(0, total - n - 1))
    result = collection.get(limit=n, offset=offset, include=["metadatas"])
    ranked = []
    for rid, meta in zip(result["ids"], result["metadatas"]):
        ranked.append(RankedResult(
            recipe_id=rid,
            name=meta.get("name", ""),
            score=0.0,
            macro_deviation=0.0,
            cost_overshoot=0.0,
            waste_fraction=0.0,
            metadata=meta,
        ))
    return ranked

In [4]:
# ── Evaluation Loop ────────────────────────────────────────────────────────────

from src.retriever import build_query_text, SearchResult

all_results = []

for profile in test_profiles:
    name = profile["name"]
    constraints = profile["constraints"]
    relevant_ids = profile["relevant_ids"]
    
    print(f"\n── {name} ──")

    # V1: random baseline (no retrieval)
    v1_results = random_baseline(collection, n=5)
    all_results.append(evaluate_variant("V1 Baseline", v1_results, constraints, relevant_ids, k=5))
    all_results[-1]["profile"] = name
    print(f"  V1 P@5={all_results[-1]['precision_at_k']:.3f}  macro_dev={all_results[-1]['macro_dev_mean_pct']:.1f}%")

    # V2: semantic search top-5
    query = build_query_text(constraints)
    v2_search = semantic_search(query, collection, model, n_results=5)
    v2_results = [RankedResult(
        recipe_id=r.recipe_id, name=r.name, score=1 - r.score,
        macro_deviation=0.0, cost_overshoot=0.0, waste_fraction=0.0,
        metadata=r.metadata
    ) for r in v2_search]
    all_results.append(evaluate_variant("V2 RAG", v2_results, constraints, relevant_ids, k=5))
    all_results[-1]["profile"] = name
    print(f"  V2 P@5={all_results[-1]['precision_at_k']:.3f}  macro_dev={all_results[-1]['macro_dev_mean_pct']:.1f}%")

    # V3: semantic search top-20 → rerank top-5
    v3_search = semantic_search(query, collection, model, n_results=20)
    v3_results = rerank(v3_search, constraints, top_k=5)
    all_results.append(evaluate_variant("V3 RAG+Rerank", v3_results, constraints, relevant_ids, k=5))
    all_results[-1]["profile"] = name
    print(f"  V3 P@5={all_results[-1]['precision_at_k']:.3f}  macro_dev={all_results[-1]['macro_dev_mean_pct']:.1f}%")

print("\nEvaluation complete.")


── Body Recomp ──
  V1 P@5=0.000  macro_dev=75.6%


  V2 P@5=0.200  macro_dev=20.9%
  V3 P@5=0.600  macro_dev=38.2%

── Muscle Bulk ──
  V1 P@5=0.000  macro_dev=71.2%
  V2 P@5=0.200  macro_dev=40.0%
  V3 P@5=0.200  macro_dev=18.6%

── Vegetarian Weight Loss ──
  V1 P@5=0.000  macro_dev=97.7%
  V2 P@5=0.200  macro_dev=60.7%
  V3 P@5=1.000  macro_dev=85.5%

── Budget Meal Prep ──
  V1 P@5=0.000  macro_dev=72.7%
  V2 P@5=0.200  macro_dev=26.2%
  V3 P@5=0.600  macro_dev=84.9%

── Keto / Low Carb ──
  V1 P@5=0.000  macro_dev=70.5%


  V2 P@5=0.600  macro_dev=45.6%
  V3 P@5=1.000  macro_dev=38.8%

Evaluation complete.


In [5]:
# ── Results DataFrame ──────────────────────────────────────────────────────────

import pandas as pd
from src.evaluator import build_results_dataframe

df = build_results_dataframe(all_results)
cols = ["profile", "variant", "precision_at_k", "macro_dev_mean_pct", "total_cost_usd", "within_budget", "waste_fraction"]
df = df[cols]

print(f"Results shape: {df.shape}  (expected 15 rows)")
df

Results shape: (15, 7)  (expected 15 rows)


,profile,variant,precision_at_k,macro_dev_mean_pct,total_cost_usd,within_budget,waste_fraction
0,Body Recomp,V1 Baseline,0.0,75.61,0.75,True,0.0
1,Body Recomp,V2 RAG,0.2,20.92,1.85,True,0.0
2,Body Recomp,V3 RAG+Rerank,0.6,38.25,3.22,True,0.0
3,Muscle Bulk,V1 Baseline,0.0,71.24,0.41,True,0.0
4,Muscle Bulk,V2 RAG,0.2,40.02,1.65,True,0.0
5,Muscle Bulk,V3 RAG+Rerank,0.2,18.63,2.53,True,0.0
6,Vegetarian Weight Loss,V1 Baseline,0.0,97.69,0.08,True,0.0
7,Vegetarian Weight Loss,V2 RAG,0.2,60.67,0.92,True,0.0
8,Vegetarian Weight Loss,V3 RAG+Rerank,1.0,85.49,1.94,True,0.0
9,Budget Meal Prep,V1 Baseline,0.0,72.65,0.68,True,0.0


In [6]:
# ── Save results ───────────────────────────────────────────────────────────────

import os
os.makedirs("../data", exist_ok=True)
df.to_csv("../data/results.csv", index=False)
print("Saved to data/results.csv")

# Quick sanity: V3 should outperform V1 on P@5 for ≥ 3 profiles
v1_p = df[df["variant"] == "V1 Baseline"].set_index("profile")["precision_at_k"]
v3_p = df[df["variant"] == "V3 RAG+Rerank"].set_index("profile")["precision_at_k"]
wins = (v3_p > v1_p).sum()
print(f"V3 > V1 on P@5: {wins}/5 profiles  (need ≥ 3)")

Saved to data/results.csv
V3 > V1 on P@5: 5/5 profiles  (need ≥ 3)
